In [1]:
import os
from dotenv import load_dotenv
from getpass import getpass
from datetime import datetime
from langfuse import get_client, Evaluation
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_qdrant import FastEmbedSparse, RetrievalMode
from qdrant_client.http.models import SearchParams

In [2]:
COLLECTION_NAME = "20260424_big_pdfs_removed"
AVG_LEN_SPARSE_FR = 255.62018086625417 # TODO : avg_len a recuperer autrement
K = 1000

load_dotenv()
langfuse = get_client()

if not os.getenv("BGE_API_KEY_EMBEDDINGS"):
    os.environ["BGE_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model (bge): ")

DATASET_NAME = "20250520_clean_dev_dataset"
RUN_NAME = datetime.now().isoformat()

dataset = langfuse.get_dataset(DATASET_NAME)

In [3]:
embeddings = OpenAIEmbeddings(
    model="BAAI/bge-m3",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("BGE_API_KEY_EMBEDDINGS")
)

sparse = FastEmbedSparse(model_name="Qdrant/bm25", avg_len= AVG_LEN_SPARSE_FR, language="french")

qdrant = QdrantVectorStore.from_existing_collection(
    url="http://localhost:6333",
    embedding=embeddings,
    sparse_embedding=sparse,
    collection_name=COLLECTION_NAME,
    retrieval_mode=RetrievalMode.HYBRID,
    vector_name="dense",
    sparse_vector_name="sparse_fr"
)

In [4]:
def retrieval_task(*, item, **kwargs):
    query = item.input.get("query")
    hits = qdrant.similarity_search_with_score(query, k=K, search_params=SearchParams(exact=True))

    retrieved_doc_ids = []
    retrieved_scores = []

    for hit in hits:
        document, score = hit
        doc_id = document.metadata.get("doc_id")
        retrieved_doc_ids.append(doc_id)
        retrieved_scores.append(score)

    return {
        "retrieved_doc_ids": retrieved_doc_ids,
        "retrieved_scores": retrieved_scores
    }

In [5]:
def make_hit_at_x_evaluator(x):
    def hit_at_x_evaluator(*, output, metadata, **kwargs):
        expected_doc_id = metadata.get("expected_doc_id")
        retrieved_doc_ids_top_k = output.get("retrieved_doc_ids")[:x]
        retrieved_scores_top_k = output.get("retrieved_scores")[:x]
        value = 1.0 if expected_doc_id in retrieved_doc_ids_top_k else 0.0
        return Evaluation(
            name=f"hit_at_{x}",
            value=value,
            comment=f"expected_doc_id={expected_doc_id}, top{x}={retrieved_doc_ids_top_k} with score {retrieved_scores_top_k}"
        )
    return hit_at_x_evaluator

def mrr_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    reciprocal_rank = 0.0
    for rank, doc_id in enumerate(retrieved_doc_ids, start=1):
        if doc_id == expected_doc_id:
            reciprocal_rank = 1.0 / rank
            break
    return Evaluation(
        name="mrr_doc",
        value=reciprocal_rank,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

def nb_correct_doc_evaluator(*, output, metadata, **kwargs):
    expected_doc_id = metadata.get("expected_doc_id")
    retrieved_doc_ids = output.get("retrieved_doc_ids")
    nb_correct_docs = 0
    for retrieved_doc_id in retrieved_doc_ids:
        if retrieved_doc_id == expected_doc_id:
            nb_correct_docs += 1
    return Evaluation(
        name="nb_correct_doc",
        value=nb_correct_docs,
        comment=f"expected_doc_id={expected_doc_id} and retrieved={retrieved_doc_ids}",
    )

In [6]:
result = dataset.run_experiment(
    name=COLLECTION_NAME,
    run_name=RUN_NAME,
    description=f"Experiment to score retrieval task on {DATASET_NAME} dataset using {COLLECTION_NAME} collection",
    task=retrieval_task,
    evaluators=[
        make_hit_at_x_evaluator(1),
        make_hit_at_x_evaluator(5),
        make_hit_at_x_evaluator(20),
        mrr_doc_evaluator,
        nb_correct_doc_evaluator
    ],
    metadata={
        "collection_name": COLLECTION_NAME,
        "top_k": K,
    }
)

print(result.format())

Individual Results: Hidden (13 items)\n💡 Set include_item_results=True to view them\n\n──────────────────────────────────────────────────\n🧪 Experiment: 20260424_big_pdfs_removed
📋 Run name: 2026-05-20T09:51:22.289996 - Experiment to score retrieval task on 20250520_clean_dev_dataset dataset using 20260424_big_pdfs_removed collection\n13 items\nEvaluations:\n  • nb_correct_doc\n  • mrr_doc\n  • hit_at_1\n  • hit_at_20\n  • hit_at_5\n\nAverage Scores:\n  • nb_correct_doc: 5.923\n  • mrr_doc: 0.376\n  • hit_at_1: 0.231\n  • hit_at_20: 0.692\n  • hit_at_5: 0.615\n\n🔗 Dataset Run:\n   http://localhost:3000/project/cmmvqmjqn0007nw07hbqsche1/datasets/cmpdor8yr0026nw07btz59oao/runs/4567fb4a-a3c3-4a32-951f-d5f151de42f3
